# DEMO PIPELINE IN CIFAR10

In [ ]:
%pip install -r requirements.txt

In [2]:
#import necessary libraries
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18
from torch.utils.data import DataLoader

# 1. LOAD DATA

In [ ]:
from deepcore.datasets import CIFAR10

#load the dataset
channel, im_size, num_classes, class_names, mean, std, dst_train, dst_test = CIFAR10('./data',False)


# 3. DEEPCORE

In [4]:
# IMPORT METHOD
from deepcore.methods import Herding

In [5]:
#dùng cái này nếu muốn sử dụng các bộ data khác mà thư viện không cung cấp(viết thay vào dst_train ở hàm select)

from torch.utils.data import Dataset
from torch.nn.functional import  one_hot


class CustomDataset(Dataset):
    def __init__(self, x:torch.tensor, y:torch.tensor, classes:list):
        """
        Args:
            x (tensor): Dữ liệu đầu vào có shape (N, H, W, C).
            y (tensor): Nhãn dạng one-hot hoặc chỉ số lớp (N,).
            classes (list): Danh sách các nhãn lớp (nếu có).
        """
        self.x = x.permute(0,3,1,2).float()  # Đổi shape thành (N, C, H, W)
        self.y = one_hot(y, num_classes=len(classes))  # Chuyển one-hot thành chỉ số lớp
        self.classes = classes 

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]


In [ ]:
#định nghĩa class args bao gồm các tham số để chạy method
class Args:
    def __init__(self, model='ResNet18', channel=3, num_classes=43, im_size=(32, 32), selection_batch=128,
                 print_freq=100, workers=4, device='cuda', selection_method="LeastConfidence",
                 selection_optimizer="Adam", selection_lr=5*1e-4, selection_momentum=0.0, selection_weight_decay=0.0,selection_nesterov=False, **kwargs):
        self.model = model
        self.channel = channel
        self.num_classes = num_classes
        self.im_size = im_size
        self.selection_batch = selection_batch
        self.print_freq = print_freq
        self.device = device
        self.selection_method = selection_method
        self.selection_optimizer = selection_optimizer
        self.selection_lr = selection_lr
        self.selection_nesterov = selection_nesterov
        self.selection_momentum = selection_momentum
        self.selection_weight_decay = selection_weight_decay
        self.gpu = None
        if device == 'cuda': 
            self.gpu = kwargs.get('gpu', None)
            self.workers = workers
        else: 
            self.workers = 0


In [ ]:
# sử dụng gpu
fraction = 0.3
args = Args(model='ResNet18', channel=3, num_classes=10, im_size=(32,32), selection_batch=1024,
            print_freq=24, workers=4, device='cuda') 
herd = Herding(dst_train=dst_train, args=args, fraction=fraction, random_seed=42, epochs=200,trainable=False)

feature_model, result = herd.select()
selected_indices = result["indices"]

In [ ]:
# sử dụng cpu
fraction = 0.3
args = Args(model='ResNet18', channel=3, num_classes=10, im_size=(32, 32), selection_batch=128,
            print_freq=100, device='cpu') 
herd = Herding(dst_train=dst_train, args=args, fraction=fraction, random_seed=1, epochs=20)

feature_model, result = herd.select()
selected_indices = result["indices"]

# 4. TEST

In [23]:
# Create a DataLoader for the selected indices and test set
from torch.utils.data import  Subset

subset_dataset = Subset(dst_train, selected_indices)

subset_dataloader = DataLoader(
    subset_dataset,
    batch_size=256,  # Use the batch size from your arguments
    shuffle=True,
    num_workers=args.workers  # Use the number of workers from your arguments
)

valloader = DataLoader(dst_test, batch_size=256, shuffle=True)

In [24]:

# Kiểm tra GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [54]:
#load model kiểu dùng thư viện DeepCore
from deepcore.nets import ResNet18
model = ResNet18(channel=3,num_classes=10,im_size=(32,32),pretrained=False)

# Loss function và optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=5*1e-4)
num_epochs = 200

In [61]:
#load model kiểu tự viết
from torchvision.models import resnet18
model = resnet18(num_classes=10).to(device)
# model = model.to(device)
# Loss function và optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=5*1e-4)
num_epochs = 200


In [62]:
#train
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for images, labels in subset_dataloader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss/len(subset_dataloader)}")

print("Training complete!")


Epoch 1, Loss: 1.8548377429024647
Epoch 2, Loss: 1.4235379392817868
Epoch 3, Loss: 1.1351247268208002
Epoch 4, Loss: 0.8794427988892894
Epoch 5, Loss: 0.6472667071778896
Epoch 6, Loss: 0.44149134068165796
Epoch 7, Loss: 0.3148197104870263
Epoch 8, Loss: 0.25436222603765585
Epoch 9, Loss: 0.1933466132919667
Epoch 10, Loss: 0.14629191665326136
Epoch 11, Loss: 0.13440782273724927
Epoch 12, Loss: 0.15813700488563312
Epoch 13, Loss: 0.13852287576360217
Epoch 14, Loss: 0.10523952587933863
Epoch 15, Loss: 0.06803790663005942
Epoch 16, Loss: 0.06219775368601589
Epoch 17, Loss: 0.07716174859364154
Epoch 18, Loss: 0.07215282106298511
Epoch 19, Loss: 0.10384853683033232
Epoch 20, Loss: 0.10226884395894358
Epoch 21, Loss: 0.08845581720441074
Epoch 22, Loss: 0.06414769937173795
Epoch 23, Loss: 0.05489078965985169
Epoch 24, Loss: 0.03432269206554708
Epoch 25, Loss: 0.02251170848701465
Epoch 26, Loss: 0.024926582524963353
Epoch 27, Loss: 0.03144030745725258
Epoch 28, Loss: 0.051962610318372814
Epoch 

In [63]:
#test
model.eval()
correct, total = 0, 0

with torch.no_grad():
    for images, labels in valloader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        # Compare predicted class indices with actual class indices (labels) directly
        correct += (predicted == labels).sum().item()  

print(f"Accuracy: {100 * correct / total:.2f}%")

Accuracy: 59.12%
